In [ ]:
import zipfile


zip_path = "DR_dataset_processed.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
"""
YOLOv11-Seg Dataset Conversion Utility
========================================
This script converts the 5-channel binary semantic masks into
YOLO standard polygon text files (instance segmentation).

Handles tiny lesions (Microaneurysms) by mathematically guaranteeing
valid polygon shapes for YOLO to learn from.
"""

import os
import cv2
import numpy as np
import pandas as pd
import shutil
from tqdm import tqdm

# Constants
BASE_DIR = 'dataset'
OUT_DIR = 'yolo_lesion_dataset'
LESION_TYPES = ['MA', 'HE', 'EX', 'OD', 'CW']
IMG_SIZE = 512.0  # YOLO normalizes coordinates to 0.0-1.0

def create_circle_polygon(cx, cy, radius, num_points=8):
    """Creates a small mathematical circle polygon around a tiny dot."""
    points = []
    for i in range(num_points):
        angle = 2 * np.pi * i / num_points
        x = cx + radius * np.cos(angle)
        y = cy + radius * np.sin(angle)
        # Keep inside image bounds
        x = np.clip(x, 0, IMG_SIZE - 1)
        y = np.clip(y, 0, IMG_SIZE - 1)
        points.append(f"{x / IMG_SIZE:.6f} {y / IMG_SIZE:.6f}")
    return " ".join(points)

def process_image(img_id, source_img_path, split_name):
    """Copies image to target dir and extracts YOLO polygons from 5 masks."""

    # Define destination folders based on CSV split
    out_img_dir = os.path.join(OUT_DIR, split_name, 'images')
    out_label_dir = os.path.join(OUT_DIR, split_name, 'labels')

    # 1. Copy image over to YOLO structure
    src_img = os.path.join(BASE_DIR, source_img_path)
    dst_img = os.path.join(out_img_dir, img_id)
    if os.path.exists(src_img):
        shutil.copy2(src_img, dst_img)

    # The label file starts empty. If no lesions are found, it stays empty (Background Image).
    label_path = os.path.join(out_label_dir, img_id.replace('.png', '.txt').replace('.jpg', '.txt'))

    yolo_lines = []

    # 2. Process all 5 lesion types
    for class_id, lt in enumerate(LESION_TYPES):
        mask_path = os.path.join(BASE_DIR, 'lesion_masks', lt, img_id)
        if not os.path.exists(mask_path):
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue

        # Extract boundaries of every white blob
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            # If the blob is extremely tiny (e.g. 1 point or line), draw a math circle
            if cnt.shape[0] < 3 or cv2.contourArea(cnt) < 4.0:
                # Find the center of the tiny dot
                M = cv2.moments(cnt)
                if M['m00'] != 0:
                    cx = M['m10'] / M['m00']
                    cy = M['m01'] / M['m00']
                else:
                    cx, cy = cnt[0][0]

                # Draw a tiny polygon so YOLO can see it (radius=3 pixels padding)
                poly_str = create_circle_polygon(cx, cy, radius=3)
                yolo_lines.append(f"{class_id} {poly_str}")

            else:
                # Standard Polygon calculation
                points = []
                for pt in cnt:
                    x, y = pt[0]
                    # Normalize against 512.0
                    points.append(f"{x / IMG_SIZE:.6f} {y / IMG_SIZE:.6f}")

                poly_str = " ".join(points)
                yolo_lines.append(f"{class_id} {poly_str}")

    # 3. Write label file
    with open(label_path, 'w') as f:
        f.write("\n".join(yolo_lines))


def main():
    # Setup directories
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(OUT_DIR, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(OUT_DIR, split, 'labels'), exist_ok=True)

    print("Starting YOLO dataset conversion...")

    splits = {
        'train': pd.read_csv(os.path.join(BASE_DIR, 'train_split.csv')),
        'val':   pd.read_csv(os.path.join(BASE_DIR, 'val_split.csv')),
        'test':  pd.read_csv(os.path.join(BASE_DIR, 'test_split.csv'))
    }

    # Process images split by split
    for split_name, df in splits.items():
        print(f"Processing {split_name} (Total: {len(df)} images)...")
        for _, row in tqdm(df.iterrows(), total=len(df)):
            process_image(row['img_id'], row['img_path'], split_name)

    # Write data.yaml config for YOLO
    yaml_content = f"""path: ../yolo_lesion_dataset  # Keep relative or absolute path based on colab mount
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: MA    # Microaneurysms
  1: HE    # Hemorrhages
  2: EX    # Hard Exudates
  3: OD    # Optic Disc
  4: CW    # Cotton Wool Spots
"""
    with open(os.path.join(OUT_DIR, 'data.yaml'), 'w') as f:
        f.write(yaml_content)

    print("✅ YOLO Dataset created successfully at:", OUT_DIR)

if __name__ == "__main__":
    main()


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# ===== SAME CONSTANTS AS YOUR SCRIPT =====
BASE_DIR = 'dataset'
OUT_DIR = 'yolo_lesion_dataset'
LESION_TYPES = ['MA', 'HE', 'EX', 'OD', 'CW']
IMG_SIZE = 512.0

CLASS_COLORS = {
    'MA': (255, 0, 0),      # red
    'HE': (0, 255, 0),      # green
    'EX': (255, 255, 0),    # yellow
    'OD': (0, 0, 255),      # blue
    'CW': (255, 0, 255)     # magenta
}

def load_split_row(img_id, split_name):
    csv_path = os.path.join(BASE_DIR, f'{split_name}_split.csv')
    df = pd.read_csv(csv_path)
    row = df[df['img_id'] == img_id]
    if row.empty:
        raise ValueError(f"Image {img_id} not found in {csv_path}")
    return row.iloc[0]

def read_image_rgb(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def make_mask_overlay(image_rgb, img_id, alpha=0.35):
    overlay = image_rgb.copy()
    instance_summary = []

    for class_id, lesion in enumerate(LESION_TYPES):
        mask_path = os.path.join(BASE_DIR, 'lesion_masks', lesion, img_id)
        if not os.path.exists(mask_path):
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue

        color = np.array(CLASS_COLORS[lesion], dtype=np.uint8)

        # color overlay where mask > 0
        colored = np.zeros_like(image_rgb, dtype=np.uint8)
        colored[mask > 0] = color
        overlay = np.where((mask[..., None] > 0),
                           (overlay * (1 - alpha) + colored * alpha).astype(np.uint8),
                           overlay)

        # contours for stats + boundary drawing
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            tiny = (cnt.shape[0] < 3) or (area < 4.0)
            instance_summary.append({
                'class_id': class_id,
                'class_name': lesion,
                'area_px': float(area),
                'points_in_contour': int(cnt.shape[0]),
                'status': 'tiny -> replaced by circle polygon' if tiny else 'normal -> kept contour'
            })
            cv2.drawContours(overlay, [cnt], -1, (255, 255, 255), 1)

    return overlay, pd.DataFrame(instance_summary)

def parse_yolo_segmentation(label_path, image_size=512):
    polygons = []
    if not os.path.exists(label_path):
        return polygons

    with open(label_path, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    for line in lines:
        parts = line.split()
        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        pts = []
        for i in range(0, len(coords), 2):
            x = coords[i] * image_size
            y = coords[i + 1] * image_size
            pts.append([x, y])

        polygons.append({
            'class_id': class_id,
            'class_name': LESION_TYPES[class_id],
            'points': np.array(pts, dtype=np.float32)
        })
    return polygons

def draw_yolo_polygons(image_rgb, polygons):
    canvas = image_rgb.copy()
    for poly in polygons:
        lesion = poly['class_name']
        color = CLASS_COLORS[lesion]
        pts = poly['points'].astype(np.int32).reshape((-1, 1, 2))
        cv2.polylines(canvas, [pts], isClosed=True, color=color, thickness=2)

        # draw first point and class label
        x0, y0 = poly['points'][0].astype(int)
        cv2.circle(canvas, (x0, y0), 3, color, -1)
        cv2.putText(canvas, lesion, (x0 + 4, y0 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return canvas

def draw_comparison_overlay(image_rgb, img_id, polygons):
    """
    Draw original contours in WHITE
    Draw YOLO polygons in lesion colors
    """
    canvas = image_rgb.copy()

    # original contours in white
    for lesion in LESION_TYPES:
        mask_path = os.path.join(BASE_DIR, 'lesion_masks', lesion, img_id)
        if not os.path.exists(mask_path):
            continue
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(canvas, contours, -1, (255, 255, 255), 1)

    # YOLO polygons in class colors
    for poly in polygons:
        lesion = poly['class_name']
        color = CLASS_COLORS[lesion]
        pts = poly['points'].astype(np.int32).reshape((-1, 1, 2))
        cv2.polylines(canvas, [pts], isClosed=True, color=color, thickness=2)

    return canvas

def visualize_conversion(img_id, split_name='train', figsize=(18, 12)):
    """
    Visualize one converted image:
      1) original image
      2) original masks overlay
      3) YOLO polygons overlay
      4) comparison overlay (original contours vs YOLO polygons)

    Also prints a summary table of what happened to each object.
    """
    row = load_split_row(img_id, split_name)
    source_img_path = os.path.join(BASE_DIR, row['img_path'])
    yolo_img_path = os.path.join(OUT_DIR, split_name, 'images', img_id)
    label_path = os.path.join(
        OUT_DIR, split_name, 'labels',
        img_id.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
    )

    # Use source image if available, otherwise use copied image
    if os.path.exists(source_img_path):
        image_rgb = read_image_rgb(source_img_path)
    elif os.path.exists(yolo_img_path):
        image_rgb = read_image_rgb(yolo_img_path)
    else:
        raise FileNotFoundError(f"Neither source nor YOLO image found for {img_id}")

    mask_overlay, summary_df = make_mask_overlay(image_rgb, img_id)
    polygons = parse_yolo_segmentation(label_path, image_size=int(IMG_SIZE))
    yolo_overlay = draw_yolo_polygons(image_rgb, polygons)
    comparison_overlay = draw_comparison_overlay(image_rgb, img_id, polygons)

    fig, axes = plt.subplots(2, 2, figsize=figsize)

    axes[0, 0].imshow(image_rgb)
    axes[0, 0].set_title("1) Original image")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(mask_overlay)
    axes[0, 1].set_title("2) Original lesion masks / contours")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(yolo_overlay)
    axes[1, 0].set_title("3) Generated YOLO polygons")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(comparison_overlay)
    axes[1, 1].set_title("4) Comparison: white=original contours, colors=YOLO polygons")
    axes[1, 1].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\nImage ID: {img_id}")
    print(f"Split: {split_name}")
    print(f"YOLO label path: {label_path}")
    print(f"Number of YOLO objects written: {len(polygons)}")

    if len(summary_df) == 0:
        print("\nNo lesions found in original masks.")
    else:
        display(summary_df)

        print("\nPer-class summary:")
        display(
            summary_df.groupby(['class_name', 'status'])
                      .size()
                      .reset_index(name='count')
                      .sort_values(['class_name', 'status'])
        )

In [ ]:
visualize_conversion("MAPLES_test_20051020_44923_0100_PP.png", split_name="train")


In [ ]:
import shutil

folder_path = 'yolo_lesion_dataset'          # your folder name
zip_name = 'yolo_lesion_dataset'      # output zip name (without .zip)

shutil.make_archive(zip_name, 'zip', folder_path)

print(f"✅ Created: {zip_name}.zip")